# Biological Age in Tohoku Medical Megabank Cohort — Biological Age Modeling with LASSO for CommCohort 67K _ cut chronic disease

NMRと500kitのmetabolomeデータ＆特定健診データが十分にある参加者をdiscovery、特定健診がない参加者をvalidationとすしてmetageモデルを作成する  
skewnessおよびimputerはdiscvoeryのものを適用

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

import sys
from decimal import Decimal, ROUND_HALF_UP
import re
#For exporting .pdf file with editable text
import matplotlib
matplotlib.rcParams['pdf.fonttype']=42
matplotlib.rcParams['ps.fonttype']=42

import collections

#!pip install MissForest
from missforest import MissForest
from sklearn.ensemble import RandomForestRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

#!conda install -c conda-forge -y watermark
%load_ext watermark
%watermark -a 'Akari FUJIMOTO' -udnt -vmw -iv

# 1. prepare metabolome data

In [ ]:
#Import data
fileDir = '../../TMM/2024-0068/20251014_TMM_CommCohort_HealthCheckup_67K/omics_data/dataTable/'
fileName = 'wholeTable.Baseline_nonPreg.n33124.c1312.20210917.wRsvNo.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t')

display(tempDF)
display(tempDF.describe(include='all'))

metDF = tempDF

In [ ]:
tempDF = metDF.copy()

#Clean
tempDF = tempDF.set_index('ID')
tempDF.index.name = 'ParticipantID'
tempL = ['Sex', 'Age', 'BMI']
tempDF1 = tempDF[tempL]
tempL = ['dateShippedSampleLIMS', 'dateNearestBloodCollection', 'reservationID', 'Dataset', 'CourseCode', 'Sex', 'Age', 'DateDiff', 'BMI']
tempDF2 = tempDF.drop(columns=tempL)

print('Metadata:')
display(tempDF1)
display(tempDF1.describe(include='all'))
print('Data:')
display(tempDF2)
display(tempDF2.describe(include='all'))

cohortDF = tempDF1
metDF = tempDF2#Update

In [ ]:
tempDF = cohortDF
variable = 'Age'
xaxis_label = 'Age [years]'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks', context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

In [ ]:
tempDF = cohortDF
variable = 'BMI'
xaxis_label = 'BMI [kg m'+r'$^{-2}$'+']'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

# 2. prepare analyte data

In [ ]:
#Import anallyte metadata
fileDir = '../../TMM/2024-0068/20251014_TMM_CommCohort_HealthCheckup_67K/omics_data/IDtable/'
fileName = 'jMorp2021.topStatsTable.20210917.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t')

display(tempDF)
display(tempDF.describe(include='all'))

analyteDF = tempDF

In [ ]:
tempDF1 = analyteDF.copy()
tempDF2 = metDF

#Clean
tempDF1 = tempDF1.set_index('ID')
tempDF1.index.name = 'AnalyteID'
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.columns)]

print('Metadata:')
display(tempDF1)
display(tempDF1.describe(include='all'))

analyteDF = tempDF1#Update

# 3. Filter based on platform

## 3-1. Select target platform analytes

Cf. readme_tcIDtable.txt  
- TCN: NMR
- TCZ: LC-MS G-Met metabolome in HILIC mode ver.1 using HPLC
- TCI: LC-MS G-Met metabolome in HILIC mode ver.2 using UHPLC
- TCO: LC-MS G-Met metabolome in C18 mode using UHPLC
- TCL: LC-MS T-Met metabolome
- TCB: LC-MS T-Met metabolome in kit180
- TCM: LC-MS T-Met metabolome in kit500
- TCS: GC-MS T-Met metabolome

In [ ]:
tempDF1 = analyteDF.copy()
tempL1 = ['TCN', 'TCM']

#Select target platform
tempDF1['PlatformPrefix'] = tempDF1.index.str[:3]
tempDF1 = tempDF1.loc[tempDF1['PlatformPrefix'].isin(tempL1)]
tempDF1 = tempDF1.sort_index(ascending=True)

display(tempDF1)
display(tempDF1.describe(include='all'))

analyteDF = tempDF1#Update

## 3-2. Select participants having the target platform data

In [ ]:
tempDF1 = metDF.copy()
tempDF2 = analyteDF

#Count non-NAs to judge whether a participant was included in the target platform measurements
tempL = []
for platform in tempDF2['PlatformPrefix'].unique():
    tempI = tempDF2.loc[tempDF2['PlatformPrefix']==platform].index
    tempDF = tempDF1[tempI]
    nrows, ncols = tempDF.shape
    tempS = tempDF.isnull().sum(axis=1)#NA count
    tempS = ncols - tempS#non-NA count
    tempS.name = 'Count_'+platform
    tempL.append(tempS)
tempDF = pd.concat(tempL, axis=1)
display(tempDF)
display(tempDF.describe(include='all'))

#Select participants having all targets
tempS = (tempDF==0).sum(axis=1)
tempS = tempS.loc[tempS==0]
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempS.index)]

#Select the target analytes
tempDF1 = tempDF1.loc[:, tempDF1.columns.isin(tempDF2.index)]

display(tempDF1)
display(tempDF1.describe(include='all'))

metDF = tempDF1#Update

In [ ]:
tempDF1 = cohortDF.copy()
tempDF2 = metDF

#Update metadata by selecting target participants
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.index)]

display(tempDF1)
display(tempDF1.describe(include='all'))

cohortDF = tempDF1#Update

In [ ]:
tempDF = cohortDF
variable = 'Age'
xaxis_label = 'Age [years]'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

In [ ]:
tempDF = cohortDF
variable = 'BMI'
xaxis_label = 'BMI [kg m'+r'$^{-2}$'+']'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

# 4. Drop columns and rows based on missingness

In [ ]:
tempDF = metDF.copy()
missing_rate = 0.1
print('DF shape before dropping columns and rows:', tempDF.shape)

#Drop columns with more missingness than cutoff
nrows, ncols = tempDF.shape
tempS = tempDF.isnull().sum(axis=0)
tempS = tempS.loc[tempS <= nrows*missing_rate]
tempDF = tempDF[tempS.index]

#Drop rows with more missingness than cutoff
nrows, ncols = tempDF.shape
tempS = tempDF.isnull().sum(axis=1)
tempS = tempS.loc[tempS <= ncols*missing_rate]
tempDF = tempDF.loc[tempS.index]

print('Baseline DF shape after dropping columns and rows:', tempDF.shape)
display(tempDF)
display(tempDF.describe(include='all'))

metDF = tempDF#Update

In [ ]:
tempDF1 = analyteDF.copy()
tempDF2 = metDF

#Update metadata by selecting target analytes
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.columns)]

display(tempDF1)
display(tempDF1.describe(include='all'))

analyteDF = tempDF1#Update

In [ ]:
tempDF1 = cohortDF.copy()
tempDF2 = metDF

#Update metadata by selecting target participants
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.index)]

display(tempDF1)
display(tempDF1.describe(include='all'))

cohortDF = tempDF1#Update

In [ ]:
tempDF = cohortDF
variable = 'Age'
xaxis_label = 'Age [years]'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

In [ ]:
tempDF = cohortDF
variable = 'BMI'
xaxis_label = 'BMI [kg m'+r'$^{-2}$'+']'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks', context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

> <– These demographic distributions are final for the defined study cohort.  

# 5. Replace analyte ID with (unique) label name

In [ ]:
tempDF1 = analyteDF.copy()

#Check multiple analyte name with different analyte IDs
tempC = collections.Counter(tempDF1['Name'])
counter = 0
for item in tempC.items():
    if item[1] > 1:
        print(item)
        counter += 1
if counter == 0:
    print('No multiple analyte name with different analyte IDs')

#Function to rename multiple analyte name
def rename_multiple(NameList):
    import collections
    count_summary = collections.Counter(NameList)
    new_list = []
    for name in NameList:
        if count_summary[name] > 1:
            counter = 1
            name_temp = f'{name} [{counter}]'
            while name_temp in new_list:
                counter += 1
                name_temp = f'{name} [{counter}]'
            new_list.append(name_temp)
        else:
            new_list.append(name)
    return new_list

#Prepare unique analyte label
tempDF1['AnalyteLabel'] = rename_multiple(tempDF1['Name'].tolist())

display(tempDF1.describe(include='all'))

analyteDF = tempDF1#Update

In [ ]:
tempDF1 = metDF.copy()
tempDF2 = analyteDF

#Replace analyte IDs with analyte labels
tempS = tempDF2['AnalyteLabel']
tempDF1.columns = tempDF1.columns.map(tempS)

display(tempDF1)
display(tempDF1.loc[:, tempDF1.columns.str.contains('\[[0-9]\]+$', regex=True)])

metDF = tempDF1#Update

In [ ]:
tempDF = metDF
tempL = ['Creatinine [1]', 'Creatinine [2]', 'Betaine [1]', 'Betaine [2]']

#Check examples of multiple measurements
sns.set(style='ticks',  context='notebook')
sns.pairplot(data=tempDF[tempL].dropna(axis=0), plot_kws={'alpha':0.5, 's':15}, height=2)
plt.show()

# 6. split special test

ここから251217版とは変化。特定健診の実施歴の有無でsplit  
特定健診あり群で歪度補正とimputationをする。  
それらのパラメータを残る群に適用させる。（ただし欠損率の確認は必要）

## 6-1. prepare special test data

In [ ]:
#Import dataframe
fileDir = './ExportData/Preprocess/'
ipynbName = '260916_TMM-BAge_CommCohort67K-ver1_'
fileName = 'specificDF-rename_Baseline-date.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

display(tempDF)
display(tempDF.describe(include='all'))
specificDF=tempDF

In [ ]:
tempDF = specificDF.copy()
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)

specificDF=tempDF

In [ ]:
spe_dict = {
    'anthopometry': [
        #anthopometry
        '身長 [cm]',
        '体重 [kg]',
        '腹囲 [cm]',
        'BMI [kg/m^2]'],

    'blood_pressure':[
        #cardiovascular
        '収縮期血圧 1回目 [mmHg]',
        '収縮期血圧 2回目 [mmHg]',
        '拡張期血圧 1回目 [mmHg]',
        '拡張期血圧 2回目 [mmHg]'],

    
    'hematopology':[
        '赤血球数 [10^4/μL]',
        '血色素量 [g/dL]',
        'ヘマトクリット値 [%]',
        '白血球数 [/μL]',
        '血小板数 [10^4/μL]',
        'リンパ球 [%]',
        '単球 [%]',
        '好酸球 [%]',
        '好塩基球 [%]',
        '好中球 [%]'],
    
    'clinical_chemistry': [
        #metabolism
        'HbA1c（NGSP） [%]',
        'グルコース [mg/dL]',
        '総コレステロール',
        '中性脂肪 [mg/dL]',
        'HDLコレステロール [mg/dL]',
        'LDLコレステロール（直接法） [mg/dL]',
        #liver
        'AST（GOT） [U/L]',
        'ALT（GPT） [U/L]',
        'γ-GTP [U/L]'],

    'kidney':[
        #renal(maybe from blood)
        '尿素窒素 [mg/dL]',
        '尿酸 [mg/dL]',
        'クレアチニン（血中） [mg/dL]',
        'GFR区分（クレアチニン）',
        #renal
        '尿蛋白',
        '尿糖',
        '尿潜血'],

    'inspection_conditions':[
        '食後区分'],

    'administrative': [
        '受診日',
        '特定健診協会名',
        '特定健診データ由来',
        '健康調査参加年度',]}


## 6-2.split discovery or validation

In [ ]:
tempDF2 = specificDF.copy()

tempDF2['受診日'] = pd.to_datetime(tempDF2['受診日'])

# まず受診日順に並べる
tempDF2 = tempDF2.sort_values('受診日', na_position='last')

# 並べ替え後の状態で重複を削除
tempDF2 = tempDF2.loc[~tempDF2.index.duplicated(keep='first')].copy()

print('Before:', len(specificDF))
print('After:', len(tempDF2))
print('Unique ID:', tempDF2.index.nunique())
print('Duplicated:', tempDF2.index.duplicated().sum())

specificDF=tempDF2

In [ ]:
tempDF1 = metDF
tempDF2 = specificDF.copy()

# metDFに情報があるID
ids = tempDF1.index

# specificDFに残った人（metあり）
tempDF2 = tempDF2.loc[tempDF2.index.isin(ids)].copy()

#specificDF index
ids = tempDF2.index

tempD2 = tempDF1.loc[tempDF1.index.isin(ids)].copy()
tempD1 = tempDF1.loc[~tempDF1.index.isin(ids)].copy()

print('Specific health check-up:', len(tempD2))
print('Not Specific health check-up:', len(tempD1))

if len(tempD2) + len(tempD1) == len(metDF):
    print('Match')
    specificDF = tempDF2
    #metDF = tempDF1
    #display(metDF)
else:
    print('Mismatch')


In [ ]:
na_rate = specificDF.isna().mean(axis=1)

print(na_rate.describe())

plt.hist(na_rate, bins=50)
plt.xlabel('Missing rate')
plt.ylabel('N')
plt.show()

In [ ]:
print('< 0.9: ',na_rate[na_rate < 0.9].describe())
print('>= 0.9: ',na_rate[na_rate >= 0.9].describe())

特定健診がある群とほぼ全欠損の群とで分割。ラインは0.9

In [ ]:
#split specific DF

# specificDF：ParticipantIDごとに1行になっている状態を想定
tempDF2 = specificDF.copy()

# ParticipantIDごとのmissing rate
na_rate = tempDF2.isna().mean(axis=1)

# 2群のIDを取得
ids_2k = na_rate.index[na_rate < 0.9]
ids_1k = na_rate.index[na_rate >= 0.9]

print('Specific health check-up:', len(ids_2k))
print('Not specific health check-up:', len(ids_1k))

In [ ]:
# metDFを2群に分割
tempDF = metDF.copy()
metDF_2k = tempDF.loc[tempDF.index.isin(ids_2k)].copy()
metDF_1k = tempDF.loc[tempDF.index.isin(ids_1k)].copy()

print('metDF:', len(metDF))
print('2k:', len(metDF_2k))
display(metDF_2k)
print('1k:', len(metDF_1k))
display(metDF_1k)
print('Total:', len(metDF_2k) + len(metDF_1k))

In [ ]:
# id duplicate check
# 2群のIDが重複していないか
print('ID overlap:', len(set(ids_2k) & set(ids_1k)))

# metDFの全IDがどちらかに入っているか
print('Unclassified:',
      len(metDF.index.difference(
          metDF_2k.index.union(metDF_1k.index)
      )))

# 2群を合わせるとmetDFと完全一致するか
print('Match:', 
      len(metDF_2k) + len(metDF_1k) == len(metDF))

In [ ]:
# cohortDFを2群に分割
tempDF = cohortDF.copy()
cohortDF_2k = tempDF.loc[tempDF.index.isin(ids_2k)].copy()
cohortDF_1k = tempDF.loc[tempDF.index.isin(ids_1k)].copy()

print('metDF:', len(cohortDF))
print('2k:', len(cohortDF_2k))
display(cohortDF_2k)
print('1k:', len(cohortDF_1k))
display(cohortDF_1k)
print('Total:', len(cohortDF_2k) + len(cohortDF_1k))

## 6-3 drop columns and rows by specific missingness

In [ ]:
tempDF = specificDF.copy()
tempDF = tempDF.drop(columns=spe_dict['administrative'],errors='ignore')
display(tempDF)

specificDF=tempDF

In [ ]:
tempDF1 = specificDF.copy()
tempDF2 = cohortDF_2k
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.index)]
display(tempDF1)

specificDF = tempDF1

In [ ]:
tempD2={}
tempDF = specificDF.copy()

for category, features in spe_dict.items():

    features = [col for col in features if col in tempDF.columns]

    resultDF = pd.DataFrame({'Feature': features,
                             'N': len(tempDF),
                             'Non-missing N': [tempDF[col].notna().sum() for col in features],
                            'missing N': [tempDF[col].isna().sum() for col in features]})
    resultDF=resultDF.sort_values('Non-missing N',ascending=False)

    print(f'===== {category} =====')
    display(resultDF)

In [ ]:
na_rate = specificDF.isna().mean(axis=1)

print(na_rate.describe())

plt.hist(na_rate, bins=50)
plt.xlabel('Missing rate')
plt.ylabel('N')
plt.show()

In [ ]:
tempDF = specificDF.copy()
missing_rate = 0.1
print('DF shape before dropping columns and rows:', tempDF.shape)

#Drop columns with more missingness than cutoff
nrows, ncols = tempDF.shape
tempS = tempDF.isnull().sum(axis=0)
tempS = tempS.loc[tempS <= nrows*missing_rate]
tempDF = tempDF[tempS.index]

#Drop rows with more missingness than cutoff
nrows, ncols = tempDF.shape
tempS = tempDF.isnull().sum(axis=1)
tempS = tempS.loc[tempS <= ncols*missing_rate]
tempDF = tempDF.loc[tempS.index]

print('Baseline DF shape after dropping columns and rows:', tempDF.shape)
display(tempDF)
display(tempDF.describe(include='all'))

specificDF = tempDF#Update

In [ ]:
tempDF2 = specificDF
tempDF1 = cohortDF_2k.copy()

#Clean
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.index)]

print('cohort data:')
display(tempDF1)
display(tempDF1.describe(include='all'))

cohortDF_2k = tempDF1#Update

In [ ]:
tempDF2 = specificDF
tempDF1 = metDF_2k.copy()

#Clean
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.index)]

print('cohort data:')
display(tempDF1)
display(tempDF1.describe(include='all'))

metDF_2k = tempDF1#Update

## 6-4 check platforms

In [ ]:
display(cohortDF_1k.describe(include='all'))

In [ ]:
tempDF1 = cohortDF_1k
tempDF2 = cohortDF_2k

variable = 'Age'
xaxis_label = 'Age [years]'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF2.groupby('Sex')[variable].describe())
display(tempDF1.groupby('Sex')[variable].describe())

sns.set(style='ticks', context='talk')
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
##Group
for ax,tempDF,title in zip(axes,[tempDF2,tempDF1],['2k','1k']):
    for group in tempD.keys():
        if group=='All':
            tempS = tempDF[variable].dropna()
        else:
            tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
        print(title, group+': skewness =', stats.skew(tempS))
        var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
        sns.distplot(tempS, label=var_label, color=tempD[group],ax=ax)
        ax.axvline(x=tempS.mean(), linestyle='--', color=tempD[group])
    sns.despine(ax=ax)
    ax.set_ylabel('Density')
    ax.set_xlabel(xaxis_label)
    ax.set_title(title)
    ax.legend(loc='upper left',labelspacing=0.25,fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
tempDF1 = cohortDF_1k
tempDF2 = cohortDF_2k

variable = 'BMI'
xaxis_label = 'BMI [kg m'+r'$^{-2}$'+']'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF2.groupby('Sex')[variable].describe())
display(tempDF1.groupby('Sex')[variable].describe())

sns.set(style='ticks', context='talk')
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
##Group
for ax,tempDF,title in zip(axes,[tempDF2,tempDF1],['2k','1k']):
    for group in tempD.keys():
        if group=='All':
            tempS = tempDF[variable].dropna()
        else:
            tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
        print(title, group+': skewness =', stats.skew(tempS))
        var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
        sns.distplot(tempS, label=var_label, color=tempD[group],ax=ax)
        ax.axvline(x=tempS.mean(), linestyle='--', color=tempD[group])
    sns.despine(ax=ax)
    ax.set_ylabel('Density')
    ax.set_xlabel(xaxis_label)
    ax.set_title(title)
    ax.legend(loc='upper right',labelspacing=0.25,fontsize=12)
plt.tight_layout()
plt.show()

# 10.save

In [ ]:
#Save cohort metadata
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'analyteDF.tsv'
tempDF = analyteDF
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

In [ ]:
#Save cohort metadata
fileDir = './ExportData/Preprocess/specific/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'specificDF.tsv'
tempDF = specificDF
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

## 10-1. 2k

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'metDF_2k-raw.tsv'
tempDF = metDF_2k
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

In [ ]:
#Save cohort metadata
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'demographicsDF_2k.tsv'
tempDF = cohortDF_2k
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

## 10-2. 1k

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'metDF_1k-raw.tsv'
tempDF = metDF_1k
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

In [ ]:
#Save cohort metadata
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'demographicsDF_1k.tsv'
tempDF = cohortDF_1k
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

# end of notebook